In [ ]:
import sys

if 'google.colab' in sys.modules:
    !pip install zombie-imp
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from pathlib import Path

if 'google.colab' in sys.modules:
    print("☁️ Environnement Colab détecté. Connexion au Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path("/content/drive/MyDrive/Documents/Code/MonPetitPronoStrategy")
    !pip install shin
else:
    print("💻 Environnement Local (VS Code) détecté.")
    PROJECT_DIR = Path.cwd().parent

sys.path.append(str(PROJECT_DIR))
DATA_DIR = PROJECT_DIR / "data"

from mpp_project.daily_pipeline import run_pf_pipeline
from mpp_project.core import load_exact_scores_by_match
from mpp_project.oracle_dp import GAP_MIN, GAP_MAX, GAP_OFFSET

# ==========================================
# PARAMÈTRES DU JOUR
# ==========================================
MATCH_DU_JOUR_ID = 73

MON_GAP_1   = 0
MON_GAP_2   = 0
HAS_BOOSTER = 1    # 1 = Booster dispo, 0 = Déjà utilisé
HORIZON_NUIT = 7   # Nombre de matchs à exporter pour l'appli mobile

# Nations des favoris — même orthographe que dans CDM_2026.csv (insensible à la casse).
# La survie de chaque favori est détectée automatiquement depuis les résultats.
MY_FAV_NATION   = "france"
BOB_FAV_NATION  = "angleterre"
PACK_FAV_NATION = "espagne"

# Scores exacts : ajoute les lignes des matchs PF (match_id >= 73) dans
# data/exact_scores.csv (colonnes : match_id, score, cote, crowd).
exact_scores_by_match = load_exact_scores_by_match(DATA_DIR / "exact_scores.csv")
print(f"🎯 Match cible : {MATCH_DU_JOUR_ID}")
print(f"📋 Matchs renseignés dans exact_scores.csv : {sorted(exact_scores_by_match)}")

In [ ]:
# ==========================================
# 1. PIPELINE PHASES FINALES (near-DP + Q-tables + abaques)
# ==========================================
import time

# Passe les scores exacts seulement si le match courant est renseigné dans le CSV.
_esbm = exact_scores_by_match if MATCH_DU_JOUR_ID in exact_scores_by_match else None

t0 = time.time()
result = run_pf_pipeline(
    csv_path        = DATA_DIR / "CDM_2026.csv",
    match_id_cible  = MATCH_DU_JOUR_ID,
    my_fav_nation   = MY_FAV_NATION,
    bob_fav_nation  = BOB_FAV_NATION,
    pack_fav_nation = PACK_FAV_NATION,
    mon_gap_1       = MON_GAP_1,
    mon_gap_2       = MON_GAP_2,
    has_booster     = HAS_BOOSTER,
    horizon_nuit    = HORIZON_NUIT,
    exact_scores_by_match = _esbm,
)

if len(result) == 5:
    reco, best_wr, market_df, Q_table_jour, night_markets = result
else:
    reco, best_wr, ev_actions, Q_table_jour = result
    night_markets = None

print(f"⏱️  Pipeline terminé en {time.time() - t0:.1f} s")
print(f"🎯 Recommandation : {reco}  |  WR : {best_wr * 100:.2f}%")

In [ ]:
# ==========================================
# 1bis. TABLEAUX SCORES EXACTS — MATCHS DE LA NUIT
# ==========================================
df_full = pd.read_csv(DATA_DIR / "CDM_2026.csv")
match_labels = {
    i + 1: f"{str(r.team_A).replace('_', ' ')} – {str(r.team_B).replace('_', ' ')}"
    for i, r in enumerate(df_full.itertuples(index=False))
}


def _afficher_marche(match_id, reco_m, wr_m, df_m):
    df_m = df_m.copy()
    if HAS_BOOSTER:
        df_m["WR best (%)"] = df_m[["WR keep (%)", "WR x2 (%)"]].max(axis=1)
    else:
        df_m["WR best (%)"] = df_m["WR base (%)"]
    view = df_m.sort_values("WR best (%)", ascending=False).reset_index(drop=True)

    fmt = {}
    for c in view.columns:
        if c == "E[pts 1/2/3]":
            fmt[c] = "{:.3f}"
        elif c.endswith("(%)") or c.startswith("E["):
            fmt[c] = "{:.2f}"
    fmt["Bonus"] = "{:.0f}"

    label = match_labels.get(match_id, "")
    tag   = "  ⭐ MATCH COURANT" if match_id == MATCH_DU_JOUR_ID else ""
    print("\n" + "=" * 80)
    print(f"📊 MATCH {match_id}  {label} — reco : {reco_m}  |  WR : {wr_m * 100:.2f}%{tag}")
    print("=" * 80)
    display(view.style.format(fmt).background_gradient(subset=["WR best (%)"], cmap="Greens"))

    agg  = df_m.groupby("Outcome")["True Proba (%)"].sum().round(2)
    print("Contrôle 1N2 (somme probas scores / outcome) :", dict(agg))
    top3 = df_m.sort_values("E[pts 1/2/3]", ascending=False).head(3)
    tops = " | ".join(f"{r['Score']} ({r['E[pts 1/2/3]']:.3f})" for _, r in top3.iterrows())
    print(f"🏅 Top 3 E[pts 1/2/3] : {tops}")


if night_markets is None:
    print("ℹ️  Mode 1N2 (aucun score exact renseigné pour le match courant).")
    print(f"🎯 Recommandation : {reco}  |  WR : {best_wr * 100:.2f}%")
elif not night_markets:
    print("⚠️  night_markets vide — vérifie exact_scores.csv.")
else:
    for mid, (reco_m, wr_m, df_m) in night_markets.items():
        _afficher_marche(mid, reco_m, wr_m, df_m)

In [ ]:
# ==========================================================================
# 1ter. DIAGNOSTIC CORRECTION 120' (PROLONGATIONS) — MATCH DU JOUR
# ==========================================================================
# MPP score à 120' alors que les cotes sont à 90'. La correction (automatique
# dans run_pf_pipeline pour les matchs KO) : Q_A (marché 'to qualify', colonnes
# cote_qualif_A/B de CDM_2026.csv) -> 1N2@120 via get_120m_outcome_probas, puis
# le multiplicateur de prolongation m est calibré pour que la grille de scores
# exacts retombe sur ce 1N2@120. Sans cote_qualif -> prior z=0.45.
from mpp_project.core import calculate_true_outcome_probas_from_odds
from mpp_project.extra_time import devig_to_qualify, build_exact_score_market_120

_df_pf = pd.read_csv(DATA_DIR / "CDM_2026.csv")
_row = _df_pf.iloc[MATCH_DU_JOUR_ID - 1]
_KO_PHASES = {"16e", "8e", "quart", "demi", "finale"}

if str(_row["phase"]).strip().lower() in _KO_PHASES and MATCH_DU_JOUR_ID in exact_scores_by_match:
    op90 = calculate_true_outcome_probas_from_odds(
        np.array([[_row.cote_1, _row.cote_N, _row.cote_2]], dtype=float))[0]
    _has_q = {"cote_qualif_A", "cote_qualif_B"} <= set(_df_pf.columns)
    Q_A = devig_to_qualify(_row.get("cote_qualif_A"), _row.get("cote_qualif_B")) if _has_q else None
    crowd120 = _row[["crowd_1", "crowd_N", "crowd_2"]].to_numpy(float)
    crowd120 = crowd120 / crowd120.sum()
    _, info = build_exact_score_market_120(
        exact_scores_by_match[MATCH_DU_JOUR_ID],
        outcome_probas_90=op90, mpp_outcome_crowd_120=crowd120, Q_A=Q_A,
    )
    qa_str = "prior z=0.45 (pas de cote to-qualify)" if Q_A is None else f"{Q_A:.1%}"
    p90, p120 = info["p1n2_90"], info["p1n2_120"]
    r90, r120 = info["region_sums_90"], info["region_sums_120"]
    print(f"⚽ Q_A (qualif {str(_row.team_A).replace('_', ' ')}) : {qa_str}")
    print(f"⏱️  Multiplicateur prolongation m calibré : {info['m']:.3f}  "
          f"(λ_A={info['lam_A']:.2f}, λ_B={info['lam_B']:.2f})")
    print(f"📊 1N2  @90'  : 1={p90[0]:.1%}  N={p90[1]:.1%}  2={p90[2]:.1%}")
    print(f"📊 1N2 @120'  : 1={p120[0]:.1%}  N={p120[1]:.1%}  2={p120[2]:.1%}   (nul déflaté)")
    print(f"🔎 Régions grille @90'  (1/N/2) : {r90[0]:.1%} / {r90[1]:.1%} / {r90[2]:.1%}")
    print(f"🔎 Régions grille @120' (1/N/2) : {r120[0]:.1%} / {r120[1]:.1%} / {r120[2]:.1%}")
else:
    print("ℹ️  Match du jour non-KO ou sans scores exacts : pas de correction 120'.")


In [ ]:
# ==========================================
# 2. PROJECTION STRATÉGIQUE (MATCH DU JOUR & SUIVANTS)
# ==========================================
print("\n" + "="*50)
print("🔮 PROJECTION STRATÉGIQUE (PHASES FINALES)")
print("="*50)
print("Analyse de sensibilité : Évolution de la stratégie selon la dynamique\n")

match_idx_pf = MATCH_DU_JOUR_ID - 73   # index 0-based dans les phases finales

noms_choix = ["1 (Dom)", "N (Nul)", "2 (Ext)"]
scenarios = [
    {"nom": "🔴 Retard (-30 pts/match)", "delta": -30},
    {"nom": "⚪ Maintien (Inchangé)",     "delta":   0},
    {"nom": "🟢 Avance (+30 pts/match)",  "delta":  30},
]

for k in range(HORIZON_NUIT):
    t_pf = match_idx_pf + k
    if t_pf >= 32:
        break

    match_id_cible_global = t_pf + 73

    try:
        abaque_path = DATA_DIR / f"Abaque_Match_{match_id_cible_global}.npz"
        data = np.load(abaque_path)
        q_table = data['q_table']
    except FileNotFoundError:
        break

    print(f"▶️ MATCH {match_id_cible_global} (Δt = +{k}) :")

    for sc in scenarios:
        proj_gap1 = MON_GAP_1 + sc["delta"] * k
        proj_gap2 = MON_GAP_2 + sc["delta"] * k

        g1_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap1)))) + GAP_OFFSET
        g2_idx = max(GAP_MIN, min(GAP_MAX, int(round(proj_gap2)))) + GAP_OFFSET

        if HAS_BOOSTER:
            wr_keep = q_table[g1_idx, g2_idx, :, 1]
            wr_use  = q_table[g1_idx, g2_idx, :, 2]
            best_keep_idx = np.argmax(wr_keep)
            best_use_idx  = np.argmax(wr_use)
            val_keep, val_use = wr_keep[best_keep_idx], wr_use[best_use_idx]
            if val_use > val_keep:
                reco_sc      = f"{noms_choix[best_use_idx]} + x2"
                details_wr = f"WR(Safe): {val_keep*100:05.2f}% | WR(x2): {val_use*100:05.2f}%"
            else:
                reco_sc      = f"{noms_choix[best_keep_idx]} (Safe)"
                details_wr = f"WR(Safe): {val_keep*100:05.2f}% | WR(x2): {val_use*100:05.2f}%"
        else:
            wr_base   = q_table[g1_idx, g2_idx, :, 0]
            best_action = np.argmax(wr_base)
            reco_sc     = f"{noms_choix[best_action]}"
            details_wr = f"WR: {wr_base[best_action]*100:05.2f}%"

        print(f"  {sc['nom']:<27} | Gaps proj. : {proj_gap1:>4}, {proj_gap2:>4} | "
              f"Reco : {reco_sc:<14} | {details_wr}")
    print("-" * 90)